# Ollama Feature Extraction

This notebook contains the `OllamaFeatureExtractor` class, which utilizes a local Ollama instance to extract product features from the product name.

In [2]:
import urllib.request
import json
import pandas as pd
import ssl
import concurrent.futures

class OllamaFeatureExtractor:
    """
    A class to interact with a local Ollama instance to perform feature extraction.
    """
    def __init__(self, model_name: str = "llama3:latest", base_url: str = "http://127.0.0.1:11434"):
        self.model_name = model_name
        self.base_url = base_url
        self.api_url = f"{self.base_url}/api/generate"

    def extract_features(self, text: str) -> str:
        """
        Sends a prompt to the local Ollama model to extract features from a product name.
        """
        prompt = f"""You are a data extraction assistant. Extract the product features from the given product name and return them as clear key-value pairs formatted nicely.\nExample output:\ndisplay - 8 inch\nHD - yes\nNetwork - Wifi\ncolor - Magenta\n\nIMPORTANT: Output ONLY the key-value pairs separated by newlines. DO NOT output any introductory or concluding conversational text.\n\nProduct Name: {text}\nFeatures:"""
        data = {
            "model": self.model_name,
            "prompt": prompt,
            "stream": False
        }
        
        try:
            req = urllib.request.Request(self.api_url, json.dumps(data).encode('utf-8'))
            req.add_header('Content-Type', 'application/json')
            # Bypass potential proxy issues that cause hangs
            handler = urllib.request.ProxyHandler({})
            opener = urllib.request.build_opener(handler)
            with opener.open(req, timeout=120) as response:
                result = json.loads(response.read().decode())
                return result.get('response', '').strip()
        except Exception as e:
            return f"Error: {e}"
            
    def analyze_dataframe(self, df: pd.DataFrame, text_column: str) -> pd.DataFrame:
        """
        Analyzes a sample of the dataframe and prepares a result dataframe.
        """
        print(f"\n--- Extracting features for products with {self.model_name} ---", flush=True)
        
        results = []
        
        def process_row(row_tuple):
            idx, row = row_tuple
            text = str(row[text_column])
            print(f"\nProcessing Product: {text[:50]}...", flush=True)
            extracted_features = self.extract_features(text)
            
            print(f"{extracted_features}\n" + "-" * 40, flush=True)
            
            # Start with original row data as a dictionary
            result_dict = row.to_dict()
            
            # Parse extracted features into individual columns
            for line in extracted_features.split('\n'):
                line = line.strip()
                if not line or "Error:" in line or "IMPORTANT:" in line or "Example output:" in line:
                    continue
                    
                # Split by delimiter '-' or ':'
                if ' - ' in line:
                    key, val = line.split(' - ', 1)
                elif ':' in line:
                    key, val = line.split(':', 1)
                else:
                    continue
                    
                # Store parsed key-value pair
                key = key.strip().lower()
                val = val.strip()
                result_dict[key] = val
                
            return result_dict
            
        with concurrent.futures.ThreadPoolExecutor(max_workers=25) as executor:
            # We use executor.map to process keeping row order
            results = list(executor.map(process_row, df.iterrows()))
            
        return pd.DataFrame(results)